In [1]:
from pathlib import Path
import os
import json
from datetime import datetime

In [3]:
from main_train import get_args
from utils.config import load_config
from data.dataset import get_vqav2
from data.dataset import get_filtered_trainval
from data.text_processing import save_tokenizer

from data.custom_generators import get_custom_generators
from models.vqa_models import get_model

from train.trainer import train_with_KD, train_from_scratch

from data.onehot_encoder import OneHotEncoder

In [4]:
config = load_config()
config['model']

{'model_architecture': 'MFBBaseline',
 'max_length': 15,
 'num_vocab_words': -1,
 'min_frequency': 5,
 'image_size': 224,
 'num_channels': 3,
 'num_classes': 1000,
 'consider_teacher': True,
 'k_window': 5,
 'output_MFB': 1024,
 'num_attention_glimps': 2,
 'embedding_dim': 100,
 'use_glove': True,
 'dropout_rate': 0}

In [5]:
config['training']

{'num_epochs': 10,
 'lr': 0.0001,
 'batch_size': 32,
 'alpha': 0.1,
 'temperature': 3,
 'knowledge_distillation': True,
 'restore_best_weights': False}

In [8]:
##### Per mio computer ############################################################
config["paths"] = {
    "dataset_path": Path("/Users/glori/Desktop/STMicroelectronics/VQA improved/vqa_dataset"),
    "glove_path": Path("/Users/glori/Desktop/STMicroelectronics/prove VQA torch/glove.6B"),
    "KD_path":  Path("/Users/glori/Desktop/STMicroelectronics/VQA KD"),
    "output_path": Path("outputs")
}

In [9]:
now = datetime.now().strftime("%y%m%d_%H%M")
saving_folder = config["paths"]["output_path"] / f"{'MFBBaseline'}_{now}"
saving_folder.mkdir(parents=True, exist_ok=True)

config["paths"]["saving_folder"] = saving_folder
config["model"]["model_architecture"] = "MFBBaseline"
# config["training"]["knowledge_distillation"] = args.distill
config["training"]["knowledge_distillation"] = True

In [10]:
saving_folder

WindowsPath('outputs/MFBBaseline_250807_1723')

In [11]:
df_train, df_val = get_filtered_trainval(
    config, consider_teacher=config["model"]["consider_teacher"], verbose=True
)

Set:  train2014
Total number of sample in train2014: 443757
Number of training samples after filtering: 388273 ( 87.50 % )
Set:  val2014
Total number of sample in val2014: 214354
Number of validation samples after filtering: 186549 ( 87.03 % )


In [24]:
##### Per mio computer ############################################################
train_im = os.listdir(config['paths']['dataset_path']/'train2014')
val_im = os.listdir(config['paths']['dataset_path']/'val2014')

df_train = df_train[df_train['image_name'].isin(train_im)]
df_val = df_val[df_val['image_name'].isin(val_im)]

len(df_train), len(df_val)

(9581, 5465)

In [25]:
len(set(df_train["normalized_answer"]))

992

In [29]:
##### Per mio computer ############################################################
config['model']['num_classes'] = len(set(df_train["normalized_answer"])) 

In [26]:
if config["model"]["num_vocab_words"] > 0:
    config["model"]["min_frequency"] = 0
    num_words = config["model"]["num_vocab_words"]
    tokenizer_path = config["paths"]["output_path"] / f"word_index{num_words}.json"
else:
    mf = config["model"]["min_frequency"]
    tokenizer_path = config["paths"]["output_path"] / f"word_index_mf{mf}.json"
if not tokenizer_path.is_file():
    #save_tokenizer(config, tokenizer_path, verbose=False)
    pass

In [27]:
tokenizer_path 

WindowsPath('outputs/word_index_mf5.json')

In [30]:
ct = 'ct' if config["model"]["consider_teacher"] else ''
num_classes = config["model"]["num_classes"]
possible_ans_path = config["paths"]["output_path"] / f"possible_answers_{ct}{num_classes}.json"
possible_ans_path

WindowsPath('outputs/possible_answers_ct992.json')

In [31]:
if not possible_ans_path.is_file():
    enc = OneHotEncoder()
    train_ans = list(df_train["normalized_answer"])
    enc.fit(train_ans)
    enc.save_json(possible_ans_path)

In [35]:
config["training"]["knowledge_distillation"] = False

In [36]:
train_data, valid_data = get_custom_generators(
    df_train, df_val, tokenizer_path, possible_ans_path, config
)

In [37]:
inp, outp, w = train_data[0]
q, im = inp
#gt, logits = outp
#q.shape, im.shape, gt.shape, logits.shape, w.shape
q.shape, im.shape, outp.shape, w.shape

((32, 15), (32, 224, 224, 3), (32, 992), (32,))

In [40]:
outp.argmax(axis=-1)

array([559, 747, 559,  31,   2, 812, 648,  17, 886,  17, 661, 987, 152,
       804, 636, 987,  17,  89, 559, 538, 480,   1, 559,   1, 987, 559,
         2, 987,   2, 559, 240, 830], dtype=int64)

In [41]:
#logits.argmax(axis=-1)

In [42]:
config['model']['model_architecture']

'MFBBaseline'

In [43]:
model = get_model(config, tokenizer_path)

Number of parameters: 24402968


In [44]:
config['training']['num_epochs'] = 2
config['training']['num_epochs']

2

In [46]:
model = train_from_scratch(model, train_data, valid_data, config)

Epoch 1/2
300/300 ━━━━━━━━━━━━━━━━━━━━ 256s 825ms/step - accuracy: 0.0058 - loss: 0.0334 - val_accuracy: 0.0073 - val_loss: 0.0307 - learning_rate: 1.0000e-04
Epoch 2/2
300/300 ━━━━━━━━━━━━━━━━━━━━ 243s 810ms/step - accuracy: 0.0908 - loss: 0.0171 - val_accuracy: 0.0106 - val_loss: 0.0278 - learning_rate: 1.0000e-04


In [47]:
arch = config["model"]["model_architecture"]
model.save(config['paths']['saving_folder'] / f"trained_{arch}.keras")

In [48]:
from train.performance import preliminary_performance

In [49]:
performance = preliminary_performance(model, train_data, valid_data, verbose=True)
with open(
    config["paths"]["saving_folder"] / "preliminary_performance.json", "w"
) as file:
    json.dump(performance, file)

300/300 ━━━━━━━━━━━━━━━━━━━━ 123s 407ms/step - accuracy: 0.1397 - loss: 0.0111
171/171 ━━━━━━━━━━━━━━━━━━━━ 69s 406ms/step - accuracy: 0.0083 - loss: 0.0269
Train Loss: 0.011580727994441986, Val Loss: 0.027789315208792686
Train Accuracy: 0.14507880806922913, Val Accuracy: 0.010612991638481617


In [51]:
f"{performance['val_accuracy']:.4f}"

'0.0106'

In [ ]:
# ora dovrei guardare se main_eval funziona ecco
# e poi boh, che altro? Cmq vedere se tutto va ok (KD va, scratch va, ma altro?)
# boh va beh direi che va tutto dai, al massimo quando torno in ufficio faccio utlime prove

In [72]:
from tensorflow import keras
from train.performance import vqa_accuracy, get_model_ans

In [53]:
from main_eval import get_vqa_accuracy

In [56]:
len(df_train), len(df_val)

(9581, 5465)

In [57]:
df_train = get_vqav2(
    config["paths"]["dataset_path"],
    split="train2014",
    keep_10ans=True,
    verbose=True,
)
df_val = get_vqav2(
    config["paths"]["dataset_path"], split="val2014", keep_10ans=True, verbose=True
)

Set:  train2014
Total number of sample in train2014: 443757
Set:  val2014
Total number of sample in val2014: 214354


In [58]:
##### Per mio computer ############################################################
train_im = os.listdir(config['paths']['dataset_path']/'train2014')
val_im = os.listdir(config['paths']['dataset_path']/'val2014')

df_train = df_train[df_train['image_name'].isin(train_im)]
df_val = df_val[df_val['image_name'].isin(val_im)]

len(df_train), len(df_val)

(10754, 6100)

In [59]:
# Extract the 10 answers associated with each question from the dataframes
train_10ans = list(df_train["normalized_10answers"])
val_10ans = list(df_val["normalized_10answers"])

In [60]:
tokenizer_path, possible_ans_path

(WindowsPath('outputs/word_index_mf5.json'),
 WindowsPath('outputs/possible_answers_ct992.json'))

In [62]:
from data.text_processing import Tokenizer

In [65]:
with open(tokenizer_path, "r", encoding="utf-8") as file:
    word_index = json.load(file)
tokenizer = Tokenizer(word_index=word_index, maxlen=config['model']['max_length'])

with open(possible_ans_path, "r", encoding="utf-8") as file:
    possible_ans = json.load(file)

In [66]:
from data.custom_generators import Custom_Generator

In [67]:
train_data = Custom_Generator(
    df_train,
    config["paths"]["dataset_path"],
    tokenizer,
    onehot_encoder=None,
    im_size=config["model"]["image_size"],
    num_channels=config["model"]["num_channels"],
    sample_weights=False,
    batch_size=config["training"]["batch_size"],
    shuffle=False,
)
valid_data = Custom_Generator(
    df_val,
    config["paths"]["dataset_path"],
    tokenizer,
    onehot_encoder=None,
    im_size=config["model"]["image_size"],
    num_channels=config["model"]["num_channels"],
    sample_weights=False,
    batch_size=config["training"]["batch_size"],
    shuffle=False,
)

In [70]:
model_path = config["paths"]["saving_folder"] / f"trained_{arch}.keras"
model_path

WindowsPath('outputs/MFBBaseline_250807_1723/trained_MFBBaseline.keras')

In [74]:
model = keras.models.load_model(model_path)

In [75]:
model_ans_train = get_model_ans(model, train_data, possible_ans)
train_accuracy = vqa_accuracy(model_ans_train, train_10ans)

In [76]:
model_ans_val = get_model_ans(model, valid_data, possible_ans)
val_accuracy = vqa_accuracy(model_ans_val, val_10ans)

In [77]:
print(
    f"Train Accuracy: {train_accuracy*100:.2f}, Val Accuracy: {val_accuracy*100:.2f}"
)

Train Accuracy: 13.01, Val Accuracy: 1.14


In [78]:
performance = {"train_accuracy": train_accuracy, "val_accuracy": val_accuracy}
with open(config["paths"]["saving_folder"] / "performance.json", "w") as file:
    json.dump(performance, file)